# NL2Bash — Google Colab

This notebook clones and runs the actual [NLP-project repository](https://github.com/Dishant402955/NLP-project). It demonstrates the classifier, deterministic entity extractor, non-executing Bash validator, committed metrics, and optional local Qwen command generation.

**Safety:** generated commands are displayed and checked only. Never add code that executes generated output. Use a standard CPU runtime: the committed local model loader configures CPU inference; a Colab GPU is not currently used.

## 1. Clone the project

In [ ]:
from pathlib import Path
import os, subprocess, sys
REPO = "https://github.com/Dishant402955/NLP-project.git"
ROOT = Path("/content/NLP-project")
if not ROOT.exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO, str(ROOT)], check=True)
elif not (ROOT / ".git").exists():
    raise RuntimeError(f"{ROOT} exists but is not a Git checkout")
else:
    dirty = subprocess.run(["git","-C",str(ROOT),"status","--porcelain"],capture_output=True,text=True,check=True).stdout.strip()
    if dirty:
        raise RuntimeError("Existing checkout has local edits. Start with a fresh Colab runtime.")
    subprocess.run(["git","-C",str(ROOT),"pull","--ff-only"],check=True)
os.chdir(ROOT)
print("Repository:", ROOT)
print("Commit:", subprocess.run(["git","rev-parse","--short","HEAD"],capture_output=True,text=True,check=True).stdout.strip())
print("Python:", sys.version)

## 2. Install the locked dependencies

The notebook exports the committed `uv.lock` dependencies and installs them into the active Colab Python kernel, then installs this repository as an editable package.

In [ ]:
subprocess.run([sys.executable,"-m","pip","install","-q","uv"],check=True)
req = Path("/content/nl2bash-colab-requirements.txt")
subprocess.run(["uv","export","--locked","--no-emit-project","--format","requirements-txt","--output-file",str(req)],cwd=ROOT,check=True)
subprocess.run(["uv","pip","install","--system","-r",str(req)],cwd=ROOT,check=True)
subprocess.run(["uv","pip","install","--system","--no-deps","-e","."],cwd=ROOT,check=True)
print("Locked dependencies and nl2bash package installed.")

## 3. Inspect the dataset and partitions

In [ ]:
import json
manifest = json.loads(Path("data/processed/nl2bash/manifest.json").read_text())
counts = {name: sum(1 for line in Path(f"data/processed/nl2bash/{name}.jsonl").open() if line.strip()) for name in ("train","dev","test")}
print("Dataset summary:")
print(json.dumps(manifest.get("summary",{}),indent=2))
print("Partition record counts:")
print(json.dumps(counts,indent=2))
with Path("data/processed/nl2bash/dev.jsonl").open() as f:
    sample=json.loads(next(f))
print("Sample request:",sample["request"])
print("Reference command (displayed for dataset inspection only):",sample["reference_command"])

## 4. Run the intent classifier

In [ ]:
from nl2bash.intent_prediction import load_intent_classifier, predict_intent
from nl2bash.evaluation import load_frozen_development_threshold
model = load_intent_classifier("models/intent_classifier.joblib","models/intent_classifier_manifest.json")
_, threshold, threshold_hash = load_frozen_development_threshold(model,"artifacts/evaluation/frozen_dev_threshold.json")
USER_REQUEST = "Find all Python files larger than 10 MB in the current directory."
prediction = predict_intent(model,USER_REQUEST,confidence_threshold=threshold)
print("Request:",USER_REQUEST)
print("Intent:",prediction.intent_label)
print("Diagnostic top label:",prediction.raw_top_label)
print("Score:",round(prediction.confidence,4))
print("Low confidence:",prediction.low_confidence)
print("Frozen threshold:",threshold)
print("Threshold SHA-256:",threshold_hash)

## 5. Extract parameters and inspect exact spans

In [ ]:
from dataclasses import asdict
from nl2bash.entity_extraction import extract_entities
extraction = extract_entities(USER_REQUEST)
print("Extracted entities:")
for item in extraction.entities:
    print(asdict(item))
    assert extraction.request[item.start:item.end] == item.value
print("\nExtraction issues:")
for issue in extraction.issues:
    print(asdict(issue))
if not extraction.issues:
    print("None")
print("\nExact source-span checks passed.")

## 6. Check Bash syntax and safety patterns (parse only)

All examples below are passed as text to the validator. They are **not executed**.

In [ ]:
from nl2bash.bash_validation import validate_bash_command
examples = {
    "safe": 'find . -type f -name "*.py"',
    "high risk": "sudo rm -rf /",
    "invalid syntax": "if then",
}
for label, command in examples.items():
    result = validate_bash_command(command)
    print(f"--- {label} ---")
    print("Command:",command)
    print("Syntax valid:",result.syntax_valid)
    print("Risk level:",result.risk_level)
    print("Review required:",result.review_required)
    for finding in result.findings:
        print(f"[{finding.severity}] {finding.code}: {finding.message}")
    print("Execution: never performed\n")

## 7. Read the committed evaluation reports

This cell reads saved JSON results. It does not rerun either one-time held-out test evaluation.

In [ ]:
def report(path):
    return json.loads(Path(path).read_text())
clf = report("artifacts/evaluation/test_report.json")["metrics"]
entity = report("artifacts/evaluation/entity_test_report.json")["metrics"]
gen = report("artifacts/evaluation/generation_dev_v7_report.json")["metrics"]
print("CLASSIFIER TEST")
print(json.dumps({k:clf[k] for k in ("record_count","accuracy","macro_f1","weighted_f1","coverage")},indent=2))
print("\nENTITY TEST")
print(json.dumps({k:entity[k] for k in ("request_count","total_gold","total_predicted","total_correct","precision","recall","f1","macro_f1")},indent=2))
print("\nGENERATION v7 DEVELOPMENT")
print(json.dumps(gen,indent=2))

## 8. Optional: install the local Qwen runtime and download the model

The runtime may compile native C++ code. The model download is approximately 1.12 GB. Set `ENABLE_LOCAL_LLM = False` to skip generation and use the notebook for the NLP and validation sections only.

In [ ]:
import hashlib, shutil, urllib.request
ENABLE_LOCAL_LLM = True
MODEL_PATH = Path("/content/qwen2.5-1.5b-instruct-q4_k_m.gguf")
MODEL_URL = "https://huggingface.co/Qwen/Qwen2.5-1.5B-Instruct-GGUF/resolve/91cad51170dc346986eccefdc2dd33a9da36ead9/qwen2.5-1.5b-instruct-q4_k_m.gguf"
EXPECTED_SHA256 = "6a1a2eb6d15622bf3c96857206351ba97e1af16c30d7a74ee38970e434e9407e"
if ENABLE_LOCAL_LLM:
    if not shutil.which("cmake") or not shutil.which("g++"):
        subprocess.run(["apt-get","update","-qq"],check=True)
        subprocess.run(["apt-get","install","-y","-qq","build-essential","cmake","python3-dev"],check=True)
    os.environ["CMAKE_ARGS"] = "-DGGML_NATIVE=OFF -DGGML_OPENMP=ON"
    os.environ["CMAKE_BUILD_PARALLEL_LEVEL"] = "2"
    os.environ["FORCE_CMAKE"] = "1"
    subprocess.run([sys.executable,"-m","pip","install","-q","--no-cache-dir","llama-cpp-python==0.3.37"],check=True)
    if not MODEL_PATH.exists():
        print("Downloading the pinned Qwen model...")
        urllib.request.urlretrieve(MODEL_URL,MODEL_PATH)
    digest=hashlib.sha256()
    with MODEL_PATH.open("rb") as f:
        for chunk in iter(lambda:f.read(8*1024*1024),b""):
            digest.update(chunk)
    actual=digest.hexdigest()
    print("Model SHA-256:",actual)
    if actual != EXPECTED_SHA256:
        MODEL_PATH.unlink(missing_ok=True)
        raise RuntimeError("Model checksum mismatch; invalid download removed.")
    print("Pinned model verified.")
else:
    print("Skipped. Set ENABLE_LOCAL_LLM=True to enable.")

## 9. Generate a Bash proposal with the complete pipeline

Edit `USER_REQUEST` in section 4 and rerun the classifier/extractor cells before this cell. This verifies the frozen configuration, loads the local model, generates a structured response, checks Bash syntax/safety, and checks selected explicit constraints. It does not execute the output.

In [ ]:
if ENABLE_LOCAL_LLM:
    from nl2bash.command_generation import load_local_generator, generate_bash_command
    from nl2bash.generation_configuration import verify_frozen_generation_configuration
    from nl2bash.entity_validation import validate_entity_grounding
    CONTEXT_SIZE, CPU_THREADS, MAX_TOKENS = 4096, 4, 512
    config_hash=verify_frozen_generation_configuration(
        "artifacts/evaluation/frozen_generation_config.json",
        model_path=MODEL_PATH,
        classifier_model_path="models/intent_classifier.joblib",
        classifier_manifest_path="models/intent_classifier_manifest.json",
        frozen_threshold_path="artifacts/evaluation/frozen_dev_threshold.json",
        context_size=CONTEXT_SIZE,threads=CPU_THREADS,max_tokens=MAX_TOKENS)
    generator=load_local_generator(MODEL_PATH,context_size=CONTEXT_SIZE,threads=CPU_THREADS)
    generated=generate_bash_command(USER_REQUEST,prediction,extraction,generator=generator,max_tokens=MAX_TOKENS)
    validation=validate_bash_command(generated.command)
    grounding=validate_entity_grounding(generated.command,extraction)
    print("PROPOSED BASH COMMAND (not executed):\n",generated.command)
    print("\nExplanation:",generated.explanation)
    print("Assumptions:",generated.assumptions or "None")
    print("\nValidation:",{"syntax_valid":validation.syntax_valid,"risk_level":validation.risk_level,
                       "review_required":validation.review_required or bool(grounding),
                       "frozen_config_sha256":config_hash,"executed":False})
    print("\nSafety findings:")
    for x in validation.findings: print(asdict(x))
    print("\nConstraint findings:")
    for x in grounding: print(asdict(x))
else:
    print("Skipped because ENABLE_LOCAL_LLM=False.")

## 10. Safety, evaluation and project notes

- The classifier test report and entity exact-span test report are one-time evaluations. Do not rerun their evaluation commands, remove their guards, or tune against those test results.
- Generation v7 is a 40-request development experiment, not held-out generation test accuracy.
- The owner confirmed human semantic review of all 80 v7 candidates. The repository records the completion attestation; row-level human scores were not supplied, so no numeric human semantic score is claimed.
- Intent/entity test labels are AI-assisted and owner-approved, not independently adjudicated gold data.
- The entity sample is targeted, not prevalence-representative.
- Syntax checks and risk rules are heuristics, not a sandbox and not proof of correctness.
- Never execute model-generated commands automatically. Review each proposal manually.

Further reading: [README](https://github.com/Dishant402955/NLP-project), [Architecture](https://github.com/Dishant402955/NLP-project/blob/main/docs/ARCHITECTURE.md), [Reproducibility Guide](https://github.com/Dishant402955/NLP-project/blob/main/docs/REPRODUCIBILITY.md), [Engineering Audit](https://github.com/Dishant402955/NLP-project/blob/main/docs/ENGINEERING_AUDIT.md).